<a href="https://colab.research.google.com/github/LamaAlghailan/multi-model-agentic-support/blob/main/02_Model_B_Technical_Extractive_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model B — Technical Extractive QA
### Tuwaiq Weekend Project: Multi-Model Agentic Technical Support System

**Task formulation:** Extractive Question Answering.

Model B receives:

`Question + Trusted Context → Start token + End token → Extracted answer`

It should be used when the exact answer exists inside trusted technical documentation.

This notebook follows the project brief:
- Base model: `distilbert-base-uncased`
- Long-context preprocessing
- `MAX_LENGTH = 384`
- `DOC_STRIDE = 96`
- At least 30 QA pairs
- Baseline before fine-tuning
- Fine-tuning
- Held-out evaluation with **Exact Match (EM)** and **token-level F1**
- Long-context boundary inspection
- Save + Hugging Face Hub verification

> **Dataset note:** The project brief says to build the QA set from supplied KB/docs. No separate support KB was provided in this chat, so this notebook uses a clearly labeled **local mock technical-support KB** for the lab. Replace these contexts with the instructor-provided KB if one is supplied later.

## 0. Install dependencies

In [1]:
!pip -q install -U transformers datasets accelerate huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.9/842.9 kB 13.2 MB/s eta 0:00:00


## 1. Imports and reproducibility

In [1]:
import os
import re
import string
import collections
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
    DefaultDataCollator,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


## 2. Build a local mock technical-support KB

Each document is a trusted context for this lab.  
The answers used in training are **literal spans inside the context**, because this is extractive QA.

We create 12 documents × 4 QA pairs = **48 QA examples**.

We split **by document**, not randomly by question, to reduce context leakage:
- Train: 8 documents = 32 QA pairs
- Validation: 2 documents = 8 QA pairs
- Test: 2 documents = 8 QA pairs

In [6]:
kb_docs = {
    "deployment": """
The support-agent service runs inside a Docker container. The application listens on port 8000,
and the container must expose port 8000. A healthy deployment should return HTTP 200 from the
/health endpoint. Environment variables are loaded from the .env file at container startup.
If the container repeatedly restarts, inspect the application logs before changing deployment settings.
""".strip(),

    "database": """
The support platform uses PostgreSQL as its primary relational database. The default maximum application
connection pool is 20 connections. When pool utilization remains above 90 percent, new requests may wait
for an available connection. Long-running transactions should be inspected before increasing the pool size.
Database migrations must be backed up and tested before production execution.
""".strip(),

    "gpu": """
The training worker uses PyTorch for GPU workloads. CUDA availability should be checked with
torch.cuda.is_available(). If CUDA is available, the model and input tensors must be placed on the same
CUDA device. CUDA out-of-memory errors should first be addressed by reducing batch size or sequence length.
The lab uses mixed precision only when the selected GPU supports it.
""".strip(),

    "api": """
The technical-support API is implemented with FastAPI. The unified chat endpoint is
/v1/chat/completions. The model discovery endpoint is /v1/models, and the service health endpoint is
/health. HTTP 422 usually indicates request validation failure, while HTTP 503 indicates that the service
is temporarily unavailable or a dependency is unhealthy.
""".strip(),

    "auth": """
The support portal uses access tokens for authenticated requests. A bearer token is sent in the
Authorization header. Expired access tokens must be refreshed before protected endpoints are called.
Repeated invalid login attempts may temporarily lock an account. MFA verification is required for users
whose accounts have multi-factor authentication enabled.
""".strip(),

    "packages": """
The Python environment should use versions recorded in requirements.txt. Transformers provides the model
and tokenizer APIs, Accelerate supports device-aware training, and PEFT provides parameter-efficient
fine-tuning utilities. bitsandbytes is used only on supported Linux/CUDA environments. Dependency conflicts
should be resolved in a clean virtual environment before deployment.
""".strip(),

    "routing": """
The production routing policy uses hard safety rules first. A fine-tuned classifier handles requests when
its confidence is above the selected threshold. Ambiguous requests below the threshold are sent to the LLM
router. Documentation questions with trusted context are routed to extractive QA. High-risk production
incidents are escalated to human support.
""".strip(),

    "observability": """
Langfuse is used for observability in the support system. A trace should record the request input,
router decision, specialist or tool calls, final answer, latency, and errors. Each request receives a unique
trace or request ID. Sensitive secrets must not be written into trace metadata.
""".strip(),

    "docker": """
The deployment stack is defined with Docker Compose. The support-agent service exposes port 8000,
while Open WebUI is available on host port 3000. PostgreSQL stores persistent data in a named volume.
The support-agent container mounts the model and data directories and reads environment variables from
the .env file.
""".strip(),

    "tickets": """
Support tickets contain an ID, title, description, priority, status, and resolution. New tickets are created
with status open. Escalated incidents use high priority and include the reason plus supporting evidence.
Ticket search returns prior cases that may contain useful resolutions. A ticket should not be closed until
the resolution has been verified.
""".strip(),

    "long_context_a": (
        "This deployment runbook contains several sections. "
        + "General checks include verifying DNS, container status, environment variables, and service logs. " * 35
        + "For the support-agent application, the required production health-check path is /health. "
        + "After checking the endpoint, operators should record the HTTP status and response time. "
        + "The service should not be restarted repeatedly without first reviewing the logs."
    ),

    "long_context_b": (
        "This database operations guide explains routine checks for PostgreSQL services. "
        + "Operators should review connection usage, lock waits, slow queries, and recent migrations. " * 35
        + "The documented warning threshold for application pool utilization is 90 percent. "
        + "If the threshold is exceeded for a sustained period, inspect long-running transactions before changing limits. "
        + "High-risk corruption cases must be escalated rather than automatically repaired."
    ),
}


# Normalize whitespace in every context BEFORE calculating answer_start.
# This converts line breaks / repeated spaces into single spaces,
# so answer_text can be matched reliably with context.index(...).
def normalize_ws(text: str) -> str:
    return " ".join(text.split())

kb_docs = {
    doc_id: normalize_ws(context)
    for doc_id, context in kb_docs.items()
}


qa_specs = {
    "deployment": [
        ("Which port must the application expose?", "8000"),
        ("Which endpoint should return HTTP 200 for a healthy deployment?", "/health"),
        ("Where are environment variables loaded from?", "the .env file"),
        ("What should be inspected when the container repeatedly restarts?", "the application logs"),
    ],
    "database": [
        ("Which database does the support platform use?", "PostgreSQL"),
        ("What is the default maximum application connection pool?", "20 connections"),
        ("Above what utilization may new requests wait for a connection?", "90 percent"),
        ("What should be inspected before increasing the pool size?", "Long-running transactions"),
    ],
    "gpu": [
        ("Which framework is used for GPU workloads?", "PyTorch"),
        ("How should CUDA availability be checked?", "torch.cuda.is_available()"),
        ("What must be on the same CUDA device?", "the model and input tensors"),
        ("What should be reduced first for CUDA out-of-memory errors?", "batch size or sequence length"),
    ],
    "api": [
        ("Which endpoint handles unified chat completions?", "/v1/chat/completions"),
        ("Which endpoint lists models?", "/v1/models"),
        ("What does HTTP 422 usually indicate?", "request validation failure,"),
        ("What does HTTP 503 indicate?", "that the service\nis temporarily unavailable or a dependency is unhealthy."),
    ],
    "auth": [
        ("Where is a bearer token sent?", "Authorization header"),
        ("What must happen to expired access tokens?", "refreshed"),
        ("What can repeated invalid login attempts do?", "temporarily lock an account"),
        ("What is required for users with multi-factor authentication enabled?", "MFA verification"),
    ],
    "packages": [
        ("Where should Python package versions be recorded?", "requirements.txt"),
        ("Which library supports device-aware training?", "Accelerate"),
        ("Which library provides parameter-efficient fine-tuning utilities?", "PEFT"),
        ("Where should dependency conflicts be resolved?", "a clean virtual environment"),
    ],
    "routing": [
        ("What does the production routing policy use first?", "hard safety rules"),
        ("When does the fine-tuned classifier handle requests?", "when its confidence is above the selected threshold"),
        ("Where are ambiguous low-confidence requests sent?", "the LLM router"),
        ("Where are documentation questions with trusted context routed?", "extractive QA"),
    ],
    "observability": [
        ("Which platform is used for observability?", "Langfuse"),
        ("What identifier does each request receive?", "a unique trace or request ID"),
        ("What should not be written into trace metadata?", "Sensitive secrets"),
        ("What should a trace record about execution time?", "latency"),
    ],
    "docker": [
        ("Which tool defines the deployment stack?", "Docker Compose"),
        ("Which port does the support-agent service expose?", "8000"),
        ("Which host port is used by Open WebUI?", "3000"),
        ("Where does the support-agent read environment variables from?", "the .env file"),
    ],
    "tickets": [
        ("What status is used for newly created tickets?", "open"),
        ("What priority is used for escalated incidents?", "high priority"),
        ("What should escalated incidents include?", "the reason plus supporting evidence"),
        ("When should a ticket be closed?", "until the resolution has been verified"),
    ],
    "long_context_a": [
        ("What is the required production health-check path?", "/health"),
        ("What should operators record after checking the endpoint?", "the HTTP status and response time"),
        ("What should not be done repeatedly before reviewing logs?", "restarted repeatedly"),
        ("Which application is this runbook for?", "support-agent"),
    ],
    "long_context_b": [
        ("What is the documented pool utilization warning threshold?", "90 percent"),
        ("What should be inspected before changing connection limits?", "long-running transactions"),
        ("Which database system is discussed in the guide?", "PostgreSQL"),
        ("What must happen in high-risk corruption cases?", "escalated"),
    ],
}

rows = []
counter = 0

for doc_id, context in kb_docs.items():
    for question, answer_text in qa_specs[doc_id]:
        # Normalize the answer the same way as the context.
        answer_text = normalize_ws(answer_text)

        # Fail with a clear message if a gold answer is not a literal span.
        if answer_text not in context:
            raise ValueError(
                f"Answer not found | doc_id={doc_id} | "
                f"answer={answer_text!r}"
            )

        answer_start = context.index(answer_text)

        rows.append({
            "id": f"qa-{counter:03d}",
            "doc_id": doc_id,
            "question": question,
            "context": context,
            "answer_text": answer_text,
            "answer_start": answer_start,
        })
        counter += 1

qa_df = pd.DataFrame(rows)

print("Total QA pairs:", len(qa_df))
print("Documents:", qa_df["doc_id"].nunique())
print(qa_df.groupby("doc_id").size())
qa_df.head()

Total QA pairs: 48
Documents: 12
doc_id
api               4
auth              4
database          4
deployment        4
docker            4
gpu               4
long_context_a    4
long_context_b    4
observability     4
packages          4
routing           4
tickets           4
dtype: int64


,id,doc_id,question,context,answer_text,answer_start
0,qa-000,deployment,Which port must the application expose?,The support-agent service runs inside a Docker...,8000,90
1,qa-001,deployment,Which endpoint should return HTTP 200 for a he...,The support-agent service runs inside a Docker...,/health,190
2,qa-002,deployment,Where are environment variables loaded from?,The support-agent service runs inside a Docker...,the .env file,246
3,qa-003,deployment,What should be inspected when the container re...,The support-agent service runs inside a Docker...,the application logs,328
4,qa-004,database,Which database does the support platform use?,The support platform uses PostgreSQL as its pr...,PostgreSQL,26


## 3. Sanity-check answer spans

For extractive QA, the character offsets must be exact.

For every row we verify:

`context[answer_start : answer_start + len(answer_text)] == answer_text`

In [7]:
for _, row in qa_df.iterrows():
    recovered = row["context"][
        row["answer_start"]:
        row["answer_start"] + len(row["answer_text"])
    ]
    assert recovered == row["answer_text"], (row["id"], recovered, row["answer_text"])

print("All answer spans are valid ✅")

All answer spans are valid ✅


## 4. Split by document

This is stronger than randomly splitting individual questions because questions from the same context do not appear in both train and test.

The test set deliberately includes one long-context document so we can inspect `DOC_STRIDE` behavior.

In [8]:
TRAIN_DOCS = [
    "deployment", "database", "gpu", "api",
    "auth", "packages", "routing", "observability",
]
VAL_DOCS = ["docker", "tickets"]
TEST_DOCS = ["long_context_a", "long_context_b"]

train_df = qa_df[qa_df["doc_id"].isin(TRAIN_DOCS)].reset_index(drop=True)
val_df = qa_df[qa_df["doc_id"].isin(VAL_DOCS)].reset_index(drop=True)
test_df = qa_df[qa_df["doc_id"].isin(TEST_DOCS)].reset_index(drop=True)

print("Train QA pairs:", len(train_df))
print("Validation QA pairs:", len(val_df))
print("Test QA pairs:", len(test_df))
print()
print("Train docs:", sorted(train_df.doc_id.unique()))
print("Val docs:", sorted(val_df.doc_id.unique()))
print("Test docs:", sorted(test_df.doc_id.unique()))

Train QA pairs: 32
Validation QA pairs: 8
Test QA pairs: 8

Train docs: ['api', 'auth', 'database', 'deployment', 'gpu', 'observability', 'packages', 'routing']
Val docs: ['docker', 'tickets']
Test docs: ['long_context_a', 'long_context_b']


## 5. Convert to Hugging Face Dataset

In [9]:
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(val_df, preserve_index=False)
test_ds = Dataset.from_pandas(test_df, preserve_index=False)

train_ds

Dataset({
    features: ['id', 'doc_id', 'question', 'context', 'answer_text', 'answer_start'],
    num_rows: 32
})

## 6. Load Model B

The brief specifies:

`distilbert-base-uncased`

with `AutoModelForQuestionAnswering`.

This adds a QA head that predicts **two logits for every token position**:
- start logit
- end logit

Before fine-tuning, this QA head has not learned our technical-support spans, so we measure a baseline first.

In [10]:
MODEL_B = "distilbert-base-uncased"

tokenizer_b = AutoTokenizer.from_pretrained(MODEL_B, use_fast=True)
model_b = AutoModelForQuestionAnswering.from_pretrained(MODEL_B)

MAX_LENGTH = 384
DOC_STRIDE = 96

print("Fast tokenizer:", tokenizer_b.is_fast)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fast tokenizer: True


# Core concept before preprocessing

Suppose we have:

**Question:** `Which port must be exposed?`

**Context:** `The application listens on port 8000.`

The model does not receive character positions directly. It receives tokens:

`[CLS] Which port ... [SEP] The application ... 8000 . [SEP]`

The training label must therefore become something like:

`start_position = token index of 8000`

`end_position = token index of 8000`

The difficult part is converting:

**character offsets in the raw context → token offsets after tokenization**

That is why we need `offset_mapping`.

## 7. Inspect `offset_mapping` on one simple example

`offset_mapping[i] = (character_start, character_end)` for token `i`.

This creates the bridge between the original string and the tokenized representation.

In [11]:
example = train_df.iloc[0]

inspection = tokenizer_b(
    example["question"],
    example["context"],
    truncation="only_second",
    max_length=MAX_LENGTH,
    return_offsets_mapping=True,
)

tokens = tokenizer_b.convert_ids_to_tokens(inspection["input_ids"])
sequence_ids = inspection.sequence_ids()

inspect_df = pd.DataFrame({
    "token_index": range(len(tokens)),
    "token": tokens,
    "sequence_id": sequence_ids,
    "offset": inspection["offset_mapping"],
})

print("Question:", example["question"])
print("Gold answer:", example["answer_text"])
print("Gold char start:", example["answer_start"])
display(inspect_df)

Question: Which port must the application expose?
Gold answer: 8000
Gold char start: 90


,token_index,token,sequence_id,offset
0,0,[CLS],NaN,"(0, 0)"
1,1,which,0.0,"(0, 5)"
2,2,port,0.0,"(6, 10)"
3,3,must,0.0,"(11, 15)"
4,4,the,0.0,"(16, 19)"
...,...,...,...,...
79,79,changing,1.0,"(356, 364)"
80,80,deployment,1.0,"(365, 375)"
81,81,settings,1.0,"(376, 384)"
82,82,.,1.0,"(384, 385)"


## 8. What is `sequence_ids()`?

The tokenizer packs two sequences together:

`[CLS] Question [SEP] Context [SEP]`

For BERT-style tokenizers, `sequence_ids()` distinguishes them:

- `None` → special token
- `0` → question
- `1` → context

We only search for the answer inside tokens where `sequence_id == 1`.

## 9. What are overflow features and `DOC_STRIDE`?

A long context may not fit inside `MAX_LENGTH=384`.

Instead of throwing away the rest, the tokenizer creates overlapping windows:

```text
Long document:
──────────────────────────────────────────────

Window 1:
[========================]

Window 2:
                  [========================]

Window 3:
                                    [========]
```

`DOC_STRIDE = 96` means neighboring context windows overlap.

`overflow_to_sample_mapping` tells us which original QA example produced each window.

In [12]:
long_example = test_df.iloc[0]

long_tokens = tokenizer_b(
    [long_example["question"]],
    [long_example["context"]],
    truncation="only_second",
    max_length=MAX_LENGTH,
    stride=DOC_STRIDE,
    return_overflowing_tokens=True,
    return_offsets_mapping=True,
    padding="max_length",
)

print("Original examples:", 1)
print("Generated features/windows:", len(long_tokens["input_ids"]))
print("overflow_to_sample_mapping:", long_tokens["overflow_to_sample_mapping"])

Original examples: 1
Generated features/windows: 2
overflow_to_sample_mapping: [0, 0]


## 10. Training preprocessing

For every generated feature/window we:

1. Find which original sample it came from.
2. Find the answer's character start/end.
3. Find the range of context tokens in this window.
4. If the answer is outside the window, assign the `[CLS]` position.
5. Otherwise convert the answer character boundaries to token start/end positions.

This is the central preprocessing step for extractive QA.

In [13]:
def prepare_train_features(examples):
    tokenized = tokenizer_b(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")
    offsets = tokenized.pop("offset_mapping")

    start_positions = []
    end_positions = []

    for feature_index, feature_offsets in enumerate(offsets):
        input_ids = tokenized["input_ids"][feature_index]
        cls_index = input_ids.index(tokenizer_b.cls_token_id)

        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]

        answer_start = examples["answer_start"][sample_index]
        answer_text = examples["answer_text"][sample_index]
        answer_end = answer_start + len(answer_text)

        # Locate the context section in this feature.
        context_start = 0
        while sequence_ids[context_start] != 1:
            context_start += 1

        context_end = len(sequence_ids) - 1
        while sequence_ids[context_end] != 1:
            context_end -= 1

        # Answer is not fully contained in this window.
        if (
            feature_offsets[context_start][0] > answer_start
            or feature_offsets[context_end][1] < answer_end
        ):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Convert answer character start to token start.
        token_start = context_start
        while (
            token_start <= context_end
            and feature_offsets[token_start][1] <= answer_start
        ):
            token_start += 1

        # Convert answer character end to token end.
        token_end = context_end
        while (
            token_end >= context_start
            and feature_offsets[token_end][0] >= answer_end
        ):
            token_end -= 1

        start_positions.append(token_start)
        end_positions.append(token_end)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions

    return tokenized

In [14]:
qa_train_features = train_ds.map(
    prepare_train_features,
    batched=True,
    remove_columns=train_ds.column_names,
)

qa_val_train_features = val_ds.map(
    prepare_train_features,
    batched=True,
    remove_columns=val_ds.column_names,
)

print("Train examples:", len(train_ds))
print("Train features/windows:", len(qa_train_features))
print("Validation examples:", len(val_ds))
print("Validation features/windows:", len(qa_val_train_features))

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Train examples: 32
Train features/windows: 32
Validation examples: 8
Validation features/windows: 8


## 11. Inspect one training label

We decode the tokens between `start_positions` and `end_positions` to confirm that the token label matches the gold answer.

In [15]:
feature = qa_train_features[0]

start = feature["start_positions"]
end = feature["end_positions"]

decoded_answer = tokenizer_b.decode(
    feature["input_ids"][start:end + 1],
    skip_special_tokens=True,
)

print("start_position:", start)
print("end_position:", end)
print("decoded training span:", decoded_answer)

start_position: 27
end_position: 28
decoded training span: 8000


## 12. Evaluation preprocessing

For EM/F1 evaluation we need to preserve:

- `example_id`
- `offset_mapping`

These let us convert predicted start/end token positions **back into text spans** in the original context.

In [16]:
def prepare_validation_features(examples):
    tokenized = tokenizer_b(
        [q.strip() for q in examples["question"]],
        examples["context"],
        truncation="only_second",
        max_length=MAX_LENGTH,
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_mapping = tokenized.pop("overflow_to_sample_mapping")

    tokenized["example_id"] = []

    for feature_index in range(len(tokenized["input_ids"])):
        sequence_ids = tokenized.sequence_ids(feature_index)
        sample_index = sample_mapping[feature_index]

        tokenized["example_id"].append(examples["id"][sample_index])

        # Keep offsets only for context tokens.
        tokenized["offset_mapping"][feature_index] = [
            offset if sequence_ids[k] == 1 else None
            for k, offset in enumerate(tokenized["offset_mapping"][feature_index])
        ]

    return tokenized

In [17]:
val_features_for_eval = val_ds.map(
    prepare_validation_features,
    batched=True,
    remove_columns=val_ds.column_names,
)

test_features_for_eval = test_ds.map(
    prepare_validation_features,
    batched=True,
    remove_columns=test_ds.column_names,
)

print("Validation eval features:", len(val_features_for_eval))
print("Test eval features:", len(test_features_for_eval))

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Validation eval features: 8
Test eval features: 16


## 13. Post-process QA logits into text answers

The model outputs:

- `start_logits`: score for each token being the answer start
- `end_logits`: score for each token being the answer end

For each original example we:
- collect all windows/features
- examine high-scoring start/end combinations
- reject invalid spans
- map the best token span back to characters using `offset_mapping`

In [18]:
def postprocess_qa_predictions(
    examples,
    features,
    raw_predictions,
    n_best_size=20,
    max_answer_length=40,
):
    start_logits, end_logits = raw_predictions

    example_id_to_index = {
        example_id: i
        for i, example_id in enumerate(examples["id"])
    }

    features_per_example = collections.defaultdict(list)

    for feature_index, feature in enumerate(features):
        features_per_example[
            example_id_to_index[feature["example_id"]]
        ].append(feature_index)

    predictions = {}

    for example_index, example in enumerate(examples):
        context = example["context"]
        candidate_answers = []

        for feature_index in features_per_example[example_index]:
            start_logit = start_logits[feature_index]
            end_logit = end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-n_best_size:][::-1]
            end_indexes = np.argsort(end_logit)[-n_best_size:][::-1]

            for start_index in start_indexes:
                for end_index in end_indexes:
                    if start_index >= len(offsets) or end_index >= len(offsets):
                        continue

                    if offsets[start_index] is None or offsets[end_index] is None:
                        continue

                    if end_index < start_index:
                        continue

                    if end_index - start_index + 1 > max_answer_length:
                        continue

                    start_char = offsets[start_index][0]
                    end_char = offsets[end_index][1]

                    candidate_answers.append({
                        "text": context[start_char:end_char],
                        "score": float(start_logit[start_index] + end_logit[end_index]),
                    })

        if candidate_answers:
            best = max(candidate_answers, key=lambda x: x["score"])
            predictions[example["id"]] = best["text"]
        else:
            predictions[example["id"]] = ""

    return predictions

## 14. Exact Match and token-level F1

We normalize answers before scoring:
- lowercase
- remove punctuation
- remove English articles (`a`, `an`, `the`)
- collapse extra whitespace

**Exact Match (EM):** normalized prediction must equal normalized gold answer exactly.

**Token F1:** measures token overlap between predicted and gold answers.

In [19]:
def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r"\b(a|an|the)\b", " ", text)

    def remove_punc(text):
        return "".join(ch for ch in text if ch not in string.punctuation)

    def white_space_fix(text):
        return " ".join(text.split())

    return white_space_fix(remove_articles(remove_punc(s.lower())))


def exact_match_score(prediction, ground_truth):
    return float(normalize_answer(prediction) == normalize_answer(ground_truth))


def token_f1_score(prediction, ground_truth):
    pred_tokens = normalize_answer(prediction).split()
    gold_tokens = normalize_answer(ground_truth).split()

    common = collections.Counter(pred_tokens) & collections.Counter(gold_tokens)
    num_same = sum(common.values())

    if len(pred_tokens) == 0 or len(gold_tokens) == 0:
        return float(pred_tokens == gold_tokens)

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gold_tokens)

    return 2 * precision * recall / (precision + recall)


def compute_qa_metrics(examples, predictions):
    em_scores = []
    f1_scores = []

    for example in examples:
        pred = predictions[example["id"]]
        gold = example["answer_text"]

        em_scores.append(exact_match_score(pred, gold))
        f1_scores.append(token_f1_score(pred, gold))

    return {
        "exact_match": float(np.mean(em_scores)),
        "token_f1": float(np.mean(f1_scores)),
    }

## 15. Helper: get raw logits from evaluation features

The evaluation feature dataset contains metadata columns that the model does not accept, so we remove them only for the prediction call.

In [20]:
def model_input_features(features):
    removable = [
        col for col in ["example_id", "offset_mapping"]
        if col in features.column_names
    ]
    return features.remove_columns(removable)

## 16. BASELINE evaluation before fine-tuning

This is required by the project.

At this point, the DistilBERT encoder is pretrained, but the QA start/end head was initialized for this task and has not been fine-tuned on our technical-support QA data.

In [21]:
baseline_args = TrainingArguments(
    output_dir="baseline_qa_tmp",
    per_device_eval_batch_size=8,
    report_to="none",
)

baseline_trainer_b = Trainer(
    model=model_b,
    args=baseline_args,
    data_collator=DefaultDataCollator(),
)

baseline_output = baseline_trainer_b.predict(
    model_input_features(test_features_for_eval)
)

baseline_predictions = postprocess_qa_predictions(
    test_ds,
    test_features_for_eval,
    baseline_output.predictions,
)

baseline_metrics = compute_qa_metrics(
    test_ds,
    baseline_predictions,
)

print("BASELINE TEST METRICS")
print(baseline_metrics)

BASELINE TEST METRICS
{'exact_match': 0.0, 'token_f1': 0.017857142857142856}


## 17. Inspect baseline predictions

In [22]:
baseline_rows = []

for ex in test_ds:
    baseline_rows.append({
        "id": ex["id"],
        "question": ex["question"],
        "gold": ex["answer_text"],
        "baseline_prediction": baseline_predictions[ex["id"]],
    })

pd.DataFrame(baseline_rows)

,id,question,gold,baseline_prediction
0,qa-040,What is the required production health-check p...,/health,", and service logs. General checks include ver..."
1,qa-041,What should operators record after checking th...,the HTTP status and response time,", and service logs. General checks include ver..."
2,qa-042,What should not be done repeatedly before revi...,restarted repeatedly,", and service logs. General checks include ver..."
3,qa-043,Which application is this runbook for?,support-agent,", and service logs. General checks include ver..."
4,qa-044,What is the documented pool utilization warnin...,90 percent,", and recent migrations. Operators should revi..."
5,qa-045,What should be inspected before changing conne...,long-running transactions,", and recent migrations. Operators should revi..."
6,qa-046,Which database system is discussed in the guide?,PostgreSQL,", and recent migrations. Operators should revi..."
7,qa-047,What must happen in high-risk corruption cases?,escalated,", and recent migrations. Operators should revi..."


## 18. Fine-tuning configuration

The brief suggests:
- learning rate `3e-5`
- batch size `8`
- `2` epochs
- best checkpoint chosen by `eval_loss`

We preserve that core setup and add light regularization.

In [23]:
args_b = TrainingArguments(
    output_dir="models/qa_model",
    learning_rate=3e-5,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    seed=SEED,
)

trainer_b = Trainer(
    model=model_b,
    args=args_b,
    train_dataset=qa_train_features,
    eval_dataset=qa_val_train_features,
    data_collator=DefaultDataCollator(),
)

## 19. Train Model B

In [24]:
train_result_b = trainer_b.train()
train_result_b

Epoch,Training Loss,Validation Loss
1,5.857514,5.723508
2,5.563793,5.620722


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8, training_loss=5.71065354347229, metrics={'train_runtime': 14.6171, 'train_samples_per_second': 4.378, 'train_steps_per_second': 0.547, 'total_flos': 6271348801536.0, 'train_loss': 5.71065354347229, 'epoch': 2.0})

## 20. Final held-out test evaluation

Important: the test set contains documents never used in training or validation.

This is where we calculate the project metrics:
- Exact Match
- token-level F1

In [25]:
test_output = trainer_b.predict(
    model_input_features(test_features_for_eval)
)

fine_tuned_predictions = postprocess_qa_predictions(
    test_ds,
    test_features_for_eval,
    test_output.predictions,
)

fine_tuned_metrics = compute_qa_metrics(
    test_ds,
    fine_tuned_predictions,
)

print("FINE-TUNED TEST METRICS")
print(fine_tuned_metrics)

FINE-TUNED TEST METRICS
{'exact_match': 0.0, 'token_f1': 0.017241379310344827}


## 21. Baseline vs fine-tuned comparison

In [26]:
comparison_b = pd.DataFrame({
    "metric": ["Exact Match", "Token F1"],
    "baseline": [
        baseline_metrics["exact_match"],
        baseline_metrics["token_f1"],
    ],
    "fine_tuned": [
        fine_tuned_metrics["exact_match"],
        fine_tuned_metrics["token_f1"],
    ],
})

comparison_b["improvement"] = (
    comparison_b["fine_tuned"] - comparison_b["baseline"]
)

comparison_b

,metric,baseline,fine_tuned,improvement
0,Exact Match,0.000000,0.000000,0.000000
1,Token F1,0.017857,0.017241,-0.000616


## 22. Error analysis + long-context boundary inspection

The brief specifically requires manual inspection of long-context boundary cases.

We display:
- question
- gold span
- predicted span
- EM
- token F1
- number of windows generated for that example

In [27]:
feature_counts = collections.Counter(test_features_for_eval["example_id"])

analysis_rows = []

for ex in test_ds:
    pred = fine_tuned_predictions[ex["id"]]
    gold = ex["answer_text"]

    analysis_rows.append({
        "id": ex["id"],
        "doc_id": ex["doc_id"],
        "question": ex["question"],
        "gold": gold,
        "prediction": pred,
        "EM": exact_match_score(pred, gold),
        "token_F1": token_f1_score(pred, gold),
        "num_windows": feature_counts[ex["id"]],
    })

analysis_df = pd.DataFrame(analysis_rows)
analysis_df

,id,doc_id,question,gold,prediction,EM,token_F1,num_windows
0,qa-040,long_context_a,What is the required production health-check p...,/health,", and service logs. General checks include ver...",0.0,0.000000,2
1,qa-041,long_context_a,What should operators record after checking th...,the HTTP status and response time,", and service logs. General checks include ver...",0.0,0.137931,2
2,qa-042,long_context_a,What should not be done repeatedly before revi...,restarted repeatedly,", and service logs. General checks include ver...",0.0,0.000000,2
3,qa-043,long_context_a,Which application is this runbook for?,support-agent,", and service logs. General checks include ver...",0.0,0.000000,2
4,qa-044,long_context_b,What is the documented pool utilization warnin...,90 percent,", and recent migrations. Operators should revi...",0.0,0.000000,2
5,qa-045,long_context_b,What should be inspected before changing conne...,long-running transactions,", and recent migrations. Operators should revi...",0.0,0.000000,2
6,qa-046,long_context_b,Which database system is discussed in the guide?,PostgreSQL,", and recent migrations. Operators should revi...",0.0,0.000000,2
7,qa-047,long_context_b,What must happen in high-risk corruption cases?,escalated,", and recent migrations. Operators should revi...",0.0,0.000000,2


## 23. Quality gate

Recommended in the project brief:

- **Exact Match ≥ 0.65**
- **Token F1 ≥ 0.80**
- manual inspection of long-context boundary cases

Do not change the threshold to force a pass. If the model fails, document the result and investigate.

In [28]:
em = fine_tuned_metrics["exact_match"]
f1 = fine_tuned_metrics["token_f1"]

gate_passed = (em >= 0.65) and (f1 >= 0.80)

print(f"Exact Match: {em:.4f}")
print(f"Token F1:   {f1:.4f}")
print("QUALITY GATE:", "PASS ✅" if gate_passed else "FAIL ❌")

Exact Match: 0.0000
Token F1:   0.0172
QUALITY GATE: FAIL ❌


## 24. Save model and tokenizer locally

In [29]:
SAVE_DIR_B = "models/qa_model"

trainer_b.save_model(SAVE_DIR_B)
tokenizer_b.save_pretrained(SAVE_DIR_B)

print("Saved Model B to:", SAVE_DIR_B)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved Model B to: models/qa_model


## 25. Reload local artifact

In [30]:
local_tokenizer_b = AutoTokenizer.from_pretrained(SAVE_DIR_B)
local_model_b = AutoModelForQuestionAnswering.from_pretrained(SAVE_DIR_B)

print("Local Model B reload successful ✅")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Local Model B reload successful ✅


## 26. Reusable single-question inference function

At production time Model B receives:
- one question
- trusted retrieved context

It returns an answer span from that context.

In [31]:
@torch.no_grad()
def answer_question(question, context, model=local_model_b, tokenizer=local_tokenizer_b):
    model.eval()

    encoded = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    offsets = encoded.pop("offset_mapping")[0]
    sequence_ids = encoded.sequence_ids(0)

    device = next(model.parameters()).device
    encoded = {k: v.to(device) for k, v in encoded.items()}

    outputs = model(**encoded)

    start_logits = outputs.start_logits[0].cpu()
    end_logits = outputs.end_logits[0].cpu()

    valid_context_positions = [
        i for i, sid in enumerate(sequence_ids)
        if sid == 1
    ]

    best_answer = ""
    best_score = -float("inf")

    for start_index in valid_context_positions:
        for end_index in valid_context_positions:
            if end_index < start_index:
                continue
            if end_index - start_index + 1 > 40:
                continue

            score = float(start_logits[start_index] + end_logits[end_index])

            if score > best_score:
                start_char = offsets[start_index][0].item() if hasattr(offsets[start_index][0], "item") else offsets[start_index][0]
                end_char = offsets[end_index][1].item() if hasattr(offsets[end_index][1], "item") else offsets[end_index][1]
                best_answer = context[start_char:end_char]
                best_score = score

    return {
        "answer": best_answer,
        "score": best_score,
    }


demo_context = kb_docs["deployment"]
demo_question = "Which port must the application expose?"

answer_question(demo_question, demo_context)

{'answer': ', and the container', 'score': 1.3753650188446045}

## 27. Upload Model B to Hugging Face Hub

Use the same secure Colab Secret pattern you used for Model A.

Create a secret named:

`colab-model-upload`

Do **not** write the token directly into the notebook.

In [32]:
from google.colab import userdata
from huggingface_hub import login, HfApi

hf_token = userdata.get("colab-model-upload")
login(token=hf_token)

api = HfApi(token=hf_token)
hf_username = api.whoami()["name"]

repo_id_b = f"{hf_username}/multi-model-support-extractive-qa"

print("Logged in as:", hf_username)
print("Model B repo:", repo_id_b)

Logged in as: Lammem310
Model B repo: Lammem310/multi-model-support-extractive-qa


In [33]:
trainer_b.model.push_to_hub(
    repo_id_b,
    token=hf_token,
)

tokenizer_b.push_to_hub(
    repo_id_b,
    token=hf_token,
)

print("Uploaded Model B ✅")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...jpf43an/model.safetensors:   0%|          |  575kB /  265MB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Uploaded Model B ✅


## 28. Reload Model B directly from Hugging Face

In [34]:
hub_tokenizer_b = AutoTokenizer.from_pretrained(
    repo_id_b,
    token=hf_token,
)

hub_model_b = AutoModelForQuestionAnswering.from_pretrained(
    repo_id_b,
    token=hf_token,
)

print("Loaded Model B from Hugging Face successfully ✅")

config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/712k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  265MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loaded Model B from Hugging Face successfully ✅


## 29. Training logs

In [35]:
history_b = pd.DataFrame(trainer_b.state.log_history)
history_b

,loss,grad_norm,learning_rate,epoch,step,eval_loss,eval_runtime,eval_samples_per_second,eval_steps_per_second,train_runtime,train_samples_per_second,train_steps_per_second,total_flos,train_loss
0,5.857514,4.984438,0.000019,1.0,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,1.0,4,5.723508,0.0741,107.958,13.495,NaN,NaN,NaN,NaN,NaN
2,5.563793,5.626445,0.000004,2.0,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,2.0,8,5.620722,0.0793,100.915,12.614,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,2.0,8,NaN,NaN,NaN,NaN,14.6171,4.378,0.547,6.271349e+12,5.710654


## 30. Model B completion checklist

Before moving to Model C:

- [ ] ≥30 SQuAD-style technical-support QA pairs
- [ ] Answers are literal spans inside trusted/mock contexts
- [ ] Dataset split by document to reduce leakage
- [ ] `offset_mapping` understood
- [ ] `sequence_ids()` understood
- [ ] `overflow_to_sample_mapping` understood
- [ ] `DOC_STRIDE` understood
- [ ] Baseline measured before fine-tuning
- [ ] Model fine-tuned
- [ ] Exact Match measured on held-out test docs
- [ ] Token-level F1 measured on held-out test docs
- [ ] Long-context boundary cases manually inspected
- [ ] Quality gate recorded honestly
- [ ] Model and tokenizer saved
- [ ] Model uploaded and reloaded from Hugging Face

### Mental model

**Training**

`Question + Context → Tokenizer → QA Encoder → start_logits + end_logits → Cross-Entropy losses → Backpropagation`

**Inference**

`Question + Trusted Context → QA Model → predicted start/end token → offset_mapping → exact text span`

**Role in the final agent**

`Router says QA → KB retrieval → Model B extracts grounded answer → return answer`